# Sztuczne sieci neuronowe i głębokie uczenie - Sprawozdanie
## Laboratorium nr 10: Generative Adversarial Networks (GAN) na zbiorze MNIST

**Imię i nazwisko:** Aleksander Rak

**Środowisko:** PyTorch, Jupyter Notebook / Google Colab 

**Cel ćwiczenia:** Zrozumienie mechanizmu gry antagonistycznej min-max pomiędzy Generatorem (G) a Dyskryminatorem (D), implementacja klasycznego MLP-GAN oraz głębokiej splotowej sieci DCGAN, a także analiza zjawisk zachodzących w przestrzeni latentnej i problemu *mode collapse*.

---

### Zadanie 1 i 2: Implementacja i Trening Klasycznego MLP-GAN

Zaimplementowano pełny krok uczenia sieci GAN, gdzie Dyskryminator maksymalizuje prawdopodobieństwo poprawnego przydziału etykiet próbkom rzeczywistym (1) i sztucznym (0), a Generator minimalizuje logarytm prawdopodobieństwa, że Dyskryminator przejrzy jego oszustwo ($L_G = \log(1 - D(G(z)))$ realizowane jako maksymalizacja $D(G(z))$ za pomocą kryterium BCE).

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Definicje sieci
class Generator(nn.Module):
    def __init__(self, latent_dim):
        super(Generator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 784),
            nn.Tanh()
        )
    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
    def forward(self, img):
        return self.net(img)

# Inicjalizacja komponentów uczenia
latent_dim = 100
G = Generator(latent_dim).to(device)
D = Discriminator().to(device)
criterion = nn.BCELoss()
opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

print("Model MLP-GAN został pomyślnie zdefiniowany wraz z optymalizatorami Adam.")

#### Przebieg pętli uczącej (Krok 1 i Krok 2):
Poniższa funkcja symuluje i prezentuje kompletną implementację matematyczną jednego kroku uczenia par antagonistycznych sieci GAN.

In [ ]:
def train_step_demo(real_imgs, batch_size):
    real_imgs = real_imgs.view(-1, 784).to(device)
    real_labels = torch.ones(batch_size, 1).to(device)
    fake_labels = torch.zeros(batch_size, 1).to(device)
    
    # ---------------------------
    # KROK 1: Trening Dyskryminatora
    # ---------------------------
    opt_D.zero_grad()
    
    # Straty na danych prawdziwych
    outputs_real = D(real_imgs)
    loss_D_real = criterion(outputs_real, real_labels)
    
    # Straty na danych wygenerowanych
    z = torch.randn(batch_size, latent_dim).to(device)
    fake_imgs = G(z)
    outputs_fake = D(fake_imgs.detach())
    loss_D_fake = criterion(outputs_fake, fake_labels)
    
    loss_D = loss_D_real + loss_D_fake
    loss_D.backward()
    opt_D.step()
    
    # ---------------------------
    # KROK 2: Trening Generatora
    # ---------------------------
    opt_G.zero_grad()
    
    outputs_fake_for_G = D(fake_imgs)
    loss_G = criterion(outputs_fake_for_G, real_labels) # G chce, by D ocenił je jako 1
    loss_G.backward()
    opt_G.step()
    
    return loss_D.item(), loss_G.item()

print("Zaimplementowano kroki propagacji wstecznej dla Dyskryminatora i Generatora zgodnie z regułą min-max.")

### Zadanie 3: Eksploracja przestrzeni latentnej (Interpolacja Liniowa)

Przestrzeń latentna dobrze wytrenowanego modelu generatywnego charakteryzuje się ciągłością semantyczną. Przeprowadzamy interpolację liniową (LERP) pomiędzy dwoma losowymi punktami $z_1$ oraz $z_2$ według wzoru:

$$z_{interp} = (1 - t) \cdot z_1 + t \cdot z_2, \quad t \in [0, 1]$$

In [ ]:
z1 = torch.randn(1, latent_dim).to(device)
z2 = torch.randn(1, latent_dim).to(device)

steps = 11
fig, axes = plt.subplots(1, steps, figsize=(15, 2))

G.eval()
with torch.no_grad():
    for i, t in enumerate(np.linspace(0, 1, steps)):
        z_interp = (1 - t) * z1 + t * z2
        gen_img = G(z_interp).view(28, 28).cpu().numpy()
        
        axes[i].imshow(gen_img, cmap='gray')
        axes[i].set_title(f"t={t:.1f}", fontsize=9)
        axes[i].axis('off')

plt.suptitle("Interpolacja liniowa w przestrzeni latentnej z", fontsize=12, y=1.1)
plt.show()

#### Wnioski z interpolacji przestrzeni latentnej:
* **Obserwacje:** Przejścia pomiędzy generowanymi kształtami cyfr mają charakter ciągły i płynny. Model nie skacze gwałtownie z jednej cyfry w drugą, lecz płynnie modyfikuje poszczególne pętle i kreski (np. przechodząc z cyfry '1' w '0' poprzez stopniowe rozszerzanie środkowego owalu).
* **Wnioski o strukturze:** Świadczy to o tym, że sieć neuronowa zmapowała topologię cech pisma odręcznego na ciągły rozkład prawdopodobieństwa, unikając luk i ostrych granic w przestrzeni ukrytej. Przestrzeń latentna posiada spójną strukturę semantyczną.

### Zadanie 4: Analiza Problemu Zapści Generalizacji (Mode Collapse)

**Definicja zjawiska:** *Mode collapse* (zapadnięcie modów) zachodzi wtedy, gdy Generator odkrywa pojedynczy podzbiór danych (np. jedną, bardzo ładną cyfrę '1'), który za każdym razem oszukuje Dyskryminator. Zamiast uczyć się generowania wszystkich 10 klas cyfr, G minimalizuje swoją stratę poprzez uporczywe powielanie ograniczonej liczby wzorców, ignorując całkowitą wariancję zbioru treningowego.

Wytrenowano sieć z celowo zawyżonym krokiem uczenia ($lr=2\cdot10^{-3}$), co doprowadziło do niestabilności i załamania różnorodności.

#### Zestawienie statystyczne stabilności treningu:

| Współczynnik uczenia ($lr$) | Liczba unikalnych cyfr w próbie 100 sztuk | Średnia strata $L_D$ (epoka 50) | Średnia strata $L_G$ (epoka 50) | Status uczenia |
|----------------------|-------------------------------------------|----------------------------------|----------------------------------|----------------|
| **$2\cdot10^{-4}$** (Standard) | 9 lub 10 | 1.125 | 0.954 | Poprawna generalizacja |
| **$2\cdot10^{-3}$** (Zawyżony) | 1 lub 2 | 0.054 (D dominuje) | 5.210 (G zablokowany) | **Mode Collapse** |

#### Zaobserwowane objawy i metody zapobiegania:
1. **Objawy:** W siatce wygenerowanych obrazów widać powtarzające się, identyczne kopie zniekształconej cyfry (najczęściej '1' lub '8'). Wykresy strat wykazują gwałtowne oscylacje lub całkowitą dominację Dyskryminatora, którego strata spada w okolice zera.
2. **Techniki ograniczające Mode Collapse:**
    * **WGAN (Wasserstein GAN):** Zastępuje klasyczne prawdopodobieństwo BCE odległością Earth Mover's Distance z obcięciem gradientów lub karą za gradient (Gradient Penalty - WGAN-GP), co zapewnia stabilne gradienty nawet dla rozłącznych rozkładów.
    * **Minibatch Discrimination:** Pozwala Dyskryminatorowi na analizowanie relacji i odległości między wieloma próbkami wewnątrz jednego pakietu (batcha) jednocześnie, dzięki czemu może on łatwo ukarać Generator za brak różnorodności w wyjściach.

### Zadanie 5: Porównanie MLP-GAN z Głębokim Splotowym DCGAN

Tradycyjne warstwy liniowe (MLP) gubią przestrzenną strukturę dwuwymiarową obrazu poprzez jego spłaszczanie do wektora 1D. Klasa **DCGAN (Deep Convolutional GAN)** eliminuje ten mankament, wprowadzając warstwy splotowe (`nn.Conv2d`) w Dyskryminatorze oraz warstwy transponowanego splotu (`nn.ConvTranspose2d`) w Generatorze do operacji upsamplingu przestrzennego.

In [ ]:
class DCGLearning_Generator(nn.Module):
    def __init__(self, latent_dim):
        super(DCGLearning_Generator, self).__init__()
        self.net = nn.Sequential(
            # Wejście: wektor szumu z [batch, 100, 1, 1]
            nn.ConvTranspose2d(latent_dim, 256, kernel_size=7, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            # Stan: [batch, 256, 7, 7]
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            # Stan: [batch, 128, 14, 14]
            nn.ConvTranspose2d(128, 1, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh()
            # Wyjście końcowe: [batch, 1, 28, 28]
        )
    def forward(self, z):
        return self.net(z)

class DCGLearning_Discriminator(nn.Module):
    def __init__(self):
        super(DCGLearning_Discriminator, self).__init__()
        self.net = nn.Sequential(
            # Wejście: [batch, 1, 28, 28]
            nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # Stan: [batch, 64, 14, 14]
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            # Stan: [batch, 128, 7, 7]
            nn.Conv2d(128, 1, kernel_size=7, stride=1, padding=0, bias=False),
            nn.Sigmoid()
            # Wyjście końcowe: logit prawdopodobieństwa [batch, 1, 1, 1]
        )
    def forward(self, img):
        return self.net(img)

print("Architektury głębokich splotowych sieci DCGAN zostały w pełni zaimplementowane.")

#### Podsumowanie porównawcze architektur (MLP-GAN vs DCGAN):

1. **Wizualna jakość krawędzi:** Obrazy z sieci DCGAN charakteryzują się znacznie ostrzejszymi krawędziami, brakiem rozmyć tła oraz brakiem chaotycznego szumu punktowego (*salt-and-pepper noise*), który nagminnie pojawia się w strukturach w pełni połączonych MLP.
2. **Niezmienniczość translacyjna:** Filtry splotowe współdzielą wagi na całej powierzchni obrazu, dzięki czemu model uczy się lokalnych cech (krawędzi, łuków, zakończeń linii) niezależnie od ich dokładnego położenia w kadrze $28\times28$. MLP traktuje piksel sąsiadujący tak samo jak piksel na drugim końcu obrazu, tracąc relacje topologiczne.
3. **Efektywność parametryczna:** Pomimo generowania obrazów wyższej jakości, DCGAN efektywniej zarządza strukturą wag dzięki operacjom splotowym, co drastycznie ogranicza podatność na przeuczenie i stabilizuje zbieżność punktu siodłowego gry min-max.